# H4 台座で渡す：情報で持ち、余裕を足して運ぶ

**ねらい**：その 1 が測った **力加減** と **重さ** を、その 2 に渡す。その 2 は、
- **運ぶ力** ＝ 力加減 × 余裕（2 倍。ただし潰れの上限の 7 割まで）で、台座から持つ
- **運ぶ時間** ＝ 1.5 秒 ×（1 ＋ 重さ ÷ 150 g）で、重い箱ほどゆっくり大箱へ運ぶ

余裕と運ぶ時間を書き換えて、**運べた／落とした／潰した** の表を作る。情報を使わずに弱く握る（わざと失敗 `weak_b`）、運ぶ高さも確かめる。
（2 台目の手先の向き `world_tilt` は H1b で済み）

計算を軽くするため、**箱が台座に置かれたところから** 始める。その 1 が見つけた力加減は、H3 の値（300 g は 0.25 N·m）を受け取る。

## 1. 準備

In [ ]:
import os, time
if "google.colab" in str(get_ipython()):
    !apt-get -qq install -y libosmesa6 fonts-noto-cjk > /dev/null
    !pip -q install genesis-world==1.4.3
    if not os.path.exists("SO-ARM100"):
        !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git && cd SO-ARM100 && git checkout -q a758567
    MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
else:   # 手元の Docker（答え合わせ用）
    MODEL = "/work/third_party/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import numpy as np, matplotlib.pyplot as plt, genesis as gs
# グラフの日本語：Colab は matplotlib を先に読み込んでいるので、入れたフォントを直接登録する（名前だけ指定しても見つからない）
from matplotlib import font_manager
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if os.path.exists(_f):
        font_manager.fontManager.addfont(_f)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]

## 2. 場面：2 台、台座、大箱
その 1 は原点（たたんだまま）、その 2 は 44 cm 先に 180° 回して置く。台座（高さ 2 cm）は真ん中、大箱（内寸 14 cm、壁の高さ 4 cm）はその 2 から見て +90°。

In [ ]:
import math
OFF = np.array([-0.0079, -0.000218, -0.0981]); FOLD = np.array([0.0, -1.7, 1.6, 1.2, 0.0, 0.0])
OPEN, CLOSE, PRE, JAW, HALF = 1.2, -0.17, 0.5, 0.019, 0.6
def rot(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])
def quat_of(R):
    t = np.trace(R)
    if t > 0:
        S = math.sqrt(t + 1.0) * 2; q = [0.25 * S, (R[2, 1] - R[1, 2]) / S, (R[0, 2] - R[2, 0]) / S, (R[1, 0] - R[0, 1]) / S]
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        S = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2; q = [(R[2, 1] - R[1, 2]) / S, 0.25 * S, (R[0, 1] + R[1, 0]) / S, (R[0, 2] + R[2, 0]) / S]
    elif R[1, 1] > R[2, 2]:
        S = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2; q = [(R[0, 2] - R[2, 0]) / S, (R[0, 1] + R[1, 0]) / S, 0.25 * S, (R[1, 2] + R[2, 1]) / S]
    else:
        S = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2; q = [(R[1, 0] - R[0, 1]) / S, (R[0, 2] + R[2, 0]) / S, (R[1, 2] + R[2, 1]) / S, 0.25 * S]
    q = np.array(q); return q / np.linalg.norm(q)
def Ry(a): return np.array([[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]])
def Rz(a): return np.array([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]])

gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10),
              rigid_options=gs.options.RigidOptions(box_box_detection=True, friction_cone=gs.friction_cone.elliptic, impratio=10.0),
              show_viewer=False)
sc.add_entity(gs.morphs.Plane())
D, STAND_H = 0.44, 0.02
entA = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(0, 0, 0), decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
entB = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(D, 0, 0), euler=(0, 0, 180), decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
sc.add_entity(gs.morphs.Box(size=(0.05, 0.05, STAND_H), pos=(D / 2, 0, STAND_H / 2), fixed=True), surface=gs.surfaces.Default(color=(0.6, 0.6, 0.65)))
BX, BY = D, -0.22                                    # 大箱の中心（その 2 から見て +90°、22 cm）
for dx, dy, sx, sy in [(0.07, 0, 0.004, 0.144), (-0.07, 0, 0.004, 0.144), (0, 0.07, 0.144, 0.004), (0, -0.07, 0.144, 0.004)]:
    sc.add_entity(gs.morphs.Box(size=(sx, sy, 0.04), pos=(BX + dx, BY + dy, 0.02), fixed=True), surface=gs.surfaces.Default(color=(0.75, 0.6, 0.4)))
box = sc.add_entity(gs.morphs.Box(size=(0.03,) * 3, pos=(D / 2, 0, STAND_H + 0.015)), material=gs.materials.Rigid(friction=1.0, rho=1000.0),
                    surface=gs.surfaces.Default(color=(0.72, 0.55, 0.36)))
cam = sc.add_camera(res=(768, 432), pos=(0.30, -0.55, 0.30), lookat=(0.32, -0.10, 0.03), fov=50, GUI=False)
t0 = time.time(); sc.build(); print(f"build {time.time() - t0:.0f} s")

class Arm:
    """根元から見た (方位 az, 半径 r, 高さ z) へ、真下向きの手先を S 字で動かす（作品の共通部品 so101.Arm を小さくしたもの）"""
    def __init__(self, ent, base, yaw_deg):
        self.e, self.g, self.base, self.yaw = ent, ent.get_link("gripper"), np.array(base, float), math.radians(yaw_deg)
        self.fixed, self.moving = self.g.idx, ent.get_link("moving_jaw_so101_v1").idx
        ent.set_dofs_position(np.zeros(6)); self.R0 = rot(self.g.get_quat().cpu().numpy())
        self.last = None
    def world(self, az, r, z):
        a = self.yaw + az; return np.array([self.base[0] + r * math.cos(a), self.base[1] + r * math.sin(a), z])
    def Rd(self, az):
        return Rz(az) @ Rz(self.yaw) @ Ry(math.pi / 2) @ Rz(-self.yaw) @ self.R0       # 腕自身の向きで真下へ（H1b）
    def tip(self): return self.g.get_pos().cpu().numpy() + rot(self.g.get_quat().cpu().numpy()) @ OFF
    def set_grip(self, tau):
        lim = np.full(6, 3.35); lim[5] = tau; self.e.set_dofs_force_range(-lim, lim)
    def jaws(self):
        c = self.e.get_contacts(with_entity=box)
        la, lb = c["link_a"].cpu().numpy(), c["link_b"].cpu().numpy()
        f = np.linalg.norm(c["force_a"].cpu().numpy(), axis=-1)
        if "valid_mask" in c: f = f * c["valid_mask"].cpu().numpy()
        return float((f * ((la == self.fixed) | (lb == self.fixed))).sum()), float((f * ((la == self.moving) | (lb == self.moving))).sum())
    def move(self, az, r, z, grip, n, every=None):
        tgt = self.world(az, r, z); start = self.last if self.last is not None else self.tip()
        for k in range(n):
            s = min(1.0, (k + 1) / (0.7 * n)); s = 3 * s * s - 2 * s ** 3
            if k % 5 == 0 or k == n - 1:
                R = self.Rd(az); cur = start + (tgt - start) * s
                q = self.e.inverse_kinematics(link=self.g, pos=cur - R @ OFF, quat=quat_of(R), init_qpos=self.e.get_dofs_position(),
                                              max_solver_iters=50).cpu().numpy()
                q[5] = grip; self.e.control_dofs_position(q)
            sc.step()
            if every: every()
        self.last = tgt
A, B = Arm(entA, (0, 0), 0), Arm(entB, (D, 0), 180)
# その 1 は、台座に置いた後、横（方位 −1.2 rad）へ退いた形でたたむ（作品と同じ）。正面でたたむと、指先が台座のすぐ手前にあり、
# 下りてくるその 2 の開いた指とぶつかる（H1b の 📕 2 と同じ）
FOLD_A = FOLD.copy(); FOLD_A[0] = -1.2

def show():
    sc.visualizer.update(force=True); plt.figure(figsize=(8, 4.5)); plt.imshow(cam.render(rgb=True)[0]); plt.axis("off"); plt.show()

def handover(mass, tau_B, carry_s, z_carry=0.07, crush=15.0, every=None):
    """台座の上の箱を、その 2 が tau_B で持ち、carry_s 秒で大箱へ運ぶ。結果の dict を返す。"""
    sc.reset()
    for arm, fold in ((A, FOLD_A), (B, FOLD)):
        arm.e.set_dofs_position(fold); arm.e.control_dofs_position(fold); arm.last = None; arm.set_grip(3.35)
    box.set_mass(mass); box.set_pos(np.array([D / 2, 0, STAND_H + 0.015])); box.set_quat(np.array([1.0, 0, 0, 0]))
    for _ in range(30): sc.step()
    st = {"peak": 0.0, "recent": [], "crushed": None, "phase": ""}
    def watch():
        st["recent"] = (st["recent"] + [min(B.jaws())])[-5:]; f5 = float(np.mean(st["recent"]))
        st["peak"] = max(st["peak"], f5)
        if f5 > crush and st["crushed"] is None: st["crushed"] = st["phase"]
        if every: every()
    r_b, zs = D / 2 - JAW, STAND_H + 0.020
    st["phase"] = "近づく"
    B.move(0.0, 0.26, 0.20, CLOSE, 120, watch)          # 高い所で手先を真下に向けてから
    B.move(0.0, r_b, 0.15, HALF, 100, watch)
    B.move(0.0, r_b, 0.15, OPEN, 40, watch)
    B.move(0.0, r_b, zs, OPEN, 120, watch)
    st["phase"] = "持つ"
    B.move(0.0, r_b, zs, PRE, 40, watch); B.set_grip(tau_B); B.move(0.0, r_b, zs, CLOSE, 200, watch)
    B.move(0.0, r_b, z_carry, CLOSE, 100, watch)
    held = float(box.get_pos().cpu().numpy()[2]) > z_carry - 0.03
    st["phase"] = "運ぶ"
    rel = np.array([BX, BY]) - B.base
    az_d = (math.atan2(rel[1], rel[0]) - B.yaw + math.pi) % (2 * math.pi) - math.pi
    B.move(az_d, float(np.linalg.norm(rel)) - JAW, z_carry, CLOSE, int(carry_s * 100), watch)
    B.move(az_d, float(np.linalg.norm(rel)) - JAW, z_carry, OPEN, 60, watch)
    B.move(az_d, float(np.linalg.norm(rel)) - JAW, 0.15, OPEN, 60, watch)
    p = box.get_pos().cpu().numpy()
    inside = abs(p[0] - BX) < 0.068 and abs(p[1] - BY) < 0.068 and p[2] < 0.06
    res = f"潰した（{st['crushed']}）" if st["crushed"] else ("運べた" if inside else ("持てない" if not held else "落とした"))
    return {"結果": res, "挟む力の最大": st["peak"], "箱": p}

FOUND, MASS = 0.25, 0.30                                    # その 1 から受け取る情報（H3 の 300 g）
def carry_tau(found, margin=2.0, crush=15.0, k=14.5):
    """運ぶ力 ＝ 力加減 × 余裕。ただし潰れの上限の 7 割（0.7 × crush ÷ k）を超えない"""
    return min(found * margin, max(0.7 * crush / k, found))
def carry_time(mass):
    return 1.5 * (1 + mass / 0.15)

r = handover(MASS, carry_tau(FOUND), carry_time(MASS))
print(f"運ぶ力 {carry_tau(FOUND):.2f} N·m   運ぶ時間 {carry_time(MASS):.1f} s   →   {r['結果']}   挟む力の最大 {r['挟む力の最大']:.1f} N")
show()

**期待される出力**：運ぶ力 0.50 N·m、運ぶ時間 4.5 s で「運べた」（挟む力の最大 8 N）。絵で、箱が大箱の中にある。

## ✏️ 変えてみる 1：情報を使わずに弱く握る（わざと失敗 `weak_b`）
**予想**：その 2 が、その 1 の情報を使わずに 0.2 N·m で握ると？

In [ ]:
r = handover(MASS, 0.2, carry_time(MASS))
print(f"運ぶ力 0.20 N·m   →   {r['結果']}"); show()

**期待される出力**：「持てない」。手だけが上がり、箱は台座に残る。

## ✏️ 変えてみる 2：余裕 × 運ぶ時間
**予想**：余裕を 1.0 倍（力加減のまま）・1.25 倍・2 倍、運ぶ時間を 0.6 秒（速い）・4.5 秒（重さに合わせる）にすると、6 通りの結果は？
1 通り 1〜2 分かかる（Colab）。

In [ ]:
print("余裕 \\ 運ぶ時間       0.6 s            4.5 s")
for margin in [1.0, 1.25, 2.0]:
    row = []
    for secs in [0.6, carry_time(MASS)]:
        r = handover(MASS, carry_tau(FOUND, margin), secs)
        row.append(f"{r['結果']}（{r['挟む力の最大']:.0f} N）")
    print(f"{margin:4.2f} 倍（{carry_tau(FOUND, margin):.2f} N·m）   " + "   ".join(f"{c:16s}" for c in row))

**期待される出力**（手元の実測）
```
1.00 倍（0.25 N·m）   落とした（4 N）   落とした（4 N）
1.25 倍（0.31 N·m）   落とした（5 N）   落とした（6 N）
2.00 倍（0.50 N·m）   運べた（11 N）    運べた（8 N）
```
この場面では、**運ぶ時間より余裕が効く**。速く運ぶと、挟む力の最大が 8 N → 11 N に上がる（振られる分、箱が指に押し付けられる）。
作品の通し（5 箱、`line.py`）では、その 1 が見つけた 300 g の力加減が 0.40 N·m で、余裕 1.0 倍でもゆっくりなら運べ、0.6 秒で運ぶと大箱の手前で落とした（動画の場面 4 の失敗）。**力加減の値しだいで、どちらが効くかが変わる**。

## ✏️ 変えてみる 3：運ぶ高さ
**予想**：運ぶ高さを 7 cm から 12 cm にすると？（H1b の届く範囲の表を思い出す）

In [ ]:
for z in [0.07, 0.12]:
    r = handover(MASS, carry_tau(FOUND), carry_time(MASS), z_carry=z)
    print(f"運ぶ高さ {z * 100:.0f} cm   →   {r['結果']}   箱 {r['箱'].round(3)}")

**期待される出力**：12 cm でも運べたが、落ちた所は大箱の中心から約 4 cm ずれた（7 cm では 0.3 cm）。

## 📕 作者の失敗の記録：その 2 が箱の上で止まった（2026-10-22）
- **何が起きたか**：この教材を書いたとき、どの条件でも「持てない」（挟む力 0）だった
- **気づいた数字**：その 2 の手先が、目標の高さ 4 cm に対して 5.9 cm で止まっていた
- **切り分け**：その 1 を正面でたたむと、指先が台座のすぐ手前（根元から 16 cm、床から 1.3 cm）にある。その 1 を横へ向けてたたむと、運べた
- **原因**：下りてくるその 2 の開いた指が、たたんだその 1 の指に当たっていた（H1b の 📕 2 と同じ形）
- **対策**：その 1 は、台座に置いた後、横（方位 −1.2 rad）へ退く（作品と同じ）
- **学び**：**2 台が同じ場所に手を伸ばす場面では、待っている側の姿勢も決める**。作品で、その 2 を台座の真上ではなく +60° で待たせたのも同じ理由（その 1 の腕とぶつかり、重さを 1606 g と読んだ）

## 確かめる問い
1. 余裕を足さずに運ぶと落とすのはなぜ？
2. 余裕を足しすぎると、何が起きる？ 作品はどう防いでいる？
3. その 2 がその 1 の情報（力加減）を使わないと、どうなる？

<details><summary>答え</summary>

1. 力加減は、2 cm を静かに持ち上げられた最小の力。運ぶ途中は、加速・減速で箱が振られ、その分の力が足りない
2. 挟む力が箱の強さを超えて潰す（段ボール 15 N）。作品は、運ぶ力を潰れの上限の 7 割（0.72 N·m）で抑える
3. 弱く握ると重い箱が持ち上がらない。強く握ると壊れやすい箱を潰す。ちょうどよい力は、持ってみないと分からない
</details>